In [14]:
import os
import numpy as np
import cv2
import nibabel as nib

from tensorflow.keras.utils import Sequence
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import classification_report


In [3]:
!pip install nibabel -q

In [4]:
IMG_SIZE = 224
BATCH_SIZE = 8

# IMPORTANT: adjust if needed
base_path = "/kaggle/input/datasets/rksrank1/pancreatic-cancer/Task07_Pancreas"

image_path = os.path.join(base_path, "imagesTr")
mask_path = os.path.join(base_path, "labelsTr")

# Get patient files
patient_files = [f for f in os.listdir(image_path) if f.endswith(".nii") and not f.startswith("._")]

print("Total patients:", len(patient_files))

Total patients: 281


In [5]:
train_files, test_files = train_test_split(patient_files, test_size=0.2, random_state=42)

print("Train patients:", len(train_files))
print("Test patients:", len(test_files))

Train patients: 224
Test patients: 57


In [6]:
class CTGenerator(Sequence):
    def __init__(self, patient_files, batch_size=BATCH_SIZE, img_size=IMG_SIZE):
        self.patient_files = patient_files
        self.batch_size = batch_size
        self.img_size = img_size
        self.samples = self._prepare_samples()

    def _prepare_samples(self):
        samples = []

        for file in self.patient_files:
            img_file = os.path.join(image_path, file)
            mask_file = os.path.join(mask_path, file)

            nii_img = nib.load(img_file)
            nii_mask = nib.load(mask_file)

            volume = nii_img.get_fdata()
            mask = nii_mask.get_fdata()

            for i in range(volume.shape[2]):
                slice_mask = mask[:, :, i]

                # Label logic
                if np.any(slice_mask == 2):
                    label = 1  # tumor
                else:
                    label = 0  # normal

                samples.append((img_file, i, label))

        np.random.shuffle(samples)
        return samples

    def __len__(self):
        return int(np.ceil(len(self.samples) / self.batch_size))

    def __getitem__(self, idx):
        batch = self.samples[idx*self.batch_size:(idx+1)*self.batch_size]

        X, y = [], []

        for file, slice_idx, label in batch:
            nii = nib.load(file)
            volume = nii.get_fdata()

            slice_img = volume[:, :, slice_idx]

            # Normalize
            slice_img = (slice_img - np.min(slice_img)) / (np.max(slice_img) + 1e-8)
            slice_img = (slice_img * 255).astype(np.uint8)

            slice_img = cv2.resize(slice_img, (self.img_size, self.img_size))
            slice_img = cv2.cvtColor(slice_img, cv2.COLOR_GRAY2RGB)

            X.append(slice_img)
            y.append(label)

        return np.array(X)/255.0, np.array(y)

In [7]:
normal_slices = 0
tumor_slices = 0

for file in train_files:
    mask = nib.load(os.path.join(mask_path, file)).get_fdata()
    tumor_slices += np.sum(np.any(mask==2, axis=(0,1)))
    normal_slices += mask.shape[2] - np.sum(np.any(mask==2, axis=(0,1)))

total = normal_slices + tumor_slices
class_weights = {
    0: total / (2 * normal_slices),
    1: total / (2 * tumor_slices)
}
print("Class weights:", class_weights)

Class weights: {0: np.float64(0.5519893355209188), 1: np.float64(5.308678500986193)}


In [8]:
train_gen = CTGenerator(train_files)
test_gen = CTGenerator(test_files)

In [9]:
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model.output
x = GlobalAveragePooling2D()(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

I0000 00:00:1774296661.727648      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [10]:
history = model.fit(
    train_gen,
    validation_data=test_gen,
    epochs=10,
    steps_per_epoch=200,
    validation_steps=50
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10


I0000 00:00:1774296699.491032     138 service.cc:152] XLA service 0x7d69900024d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1774296699.491163     138 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1774296704.156232     138 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1774296724.378849     138 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


200/200 ━━━━━━━━━━━━━━━━━━━━ 469s 2s/step - accuracy: 0.8523 - loss: 0.4078 - val_accuracy: 0.8750 - val_loss: 1.1568
Epoch 2/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 324s 2s/step - accuracy: 0.8696 - loss: 0.3304 - val_accuracy: 0.8775 - val_loss: 0.4127
Epoch 3/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 226s 1s/step - accuracy: 0.9077 - loss: 0.2712 - val_accuracy: 0.8750 - val_loss: 0.5745
Epoch 4/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 283s 1s/step - accuracy: 0.9032 - loss: 0.2962 - val_accuracy: 0.8750 - val_loss: 1.5459
Epoch 5/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 370s 2s/step - accuracy: 0.8975 - loss: 0.3023 - val_accuracy: 0.8750 - val_loss: 0.5948
Epoch 6/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 278s 1s/step - accuracy: 0.9071 - loss: 0.2748 - val_accuracy: 0.8750 - val_loss: 1.2488
Epoch 7/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 371s 2s/step - accuracy: 0.8992 - loss: 0.2723 - val_accuracy: 0.8750 - val_loss: 0.3896
Epoch 8/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 378s 2s/step - accuracy: 0.9062 - loss: 0.2648 - val_accuracy: 0.875

In [11]:
def predict_patient(model, patient_file):
    nii_img = nib.load(os.path.join(image_path, patient_file)).get_fdata()
    mask = nib.load(os.path.join(mask_path, patient_file)).get_fdata()
    
    preds = []
    for i in range(nii_img.shape[2]):
        slice_img = nii_img[:, :, i]
        slice_img = (slice_img - np.min(slice_img)) / (np.max(slice_img)+1e-8)
        slice_img = (slice_img*255).astype(np.uint8)
        slice_img = cv2.resize(slice_img, (IMG_SIZE, IMG_SIZE))
        slice_img = cv2.cvtColor(slice_img, cv2.COLOR_GRAY2RGB)
        slice_img = np.expand_dims(slice_img/255.0, axis=0)
        pred = model.predict(slice_img, verbose=0)[0][0]
        preds.append(pred)
    # Aggregate slice predictions to patient-level
    return 1 if np.mean(preds) > 0.5 else 0

In [15]:
y_true, y_pred = [], []

for file in test_files:
    mask = nib.load(os.path.join(mask_path, file)).get_fdata()
    y_true.append(1 if np.any(mask==2) else 0)
    y_pred.append(predict_patient(model, file))

print(classification_report(y_true, y_pred))
print("ROC-AUC:", roc_auc_score(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       0.0
           1       0.00      0.00      0.00      57.0

    accuracy                           0.00      57.0
   macro avg       0.00      0.00      0.00      57.0
weighted avg       0.00      0.00      0.00      57.0

ROC-AUC: nan


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

In [17]:
# Save model after training
model.save('/kaggle/working/pancreas_model.h5')

In [19]:
import numpy as np
np.save('/kaggle/working/X.npy', X)
np.save('/kaggle/working/y.npy', y)

NameError: name 'X' is not defined